# Activity 4: RAG From Scratch, Then Chroma, Then LlamaIndex

**Week 6 Day 4 | The retrieval you built, the generation you know, and what a framework actually replaces**

**Estimated time:** 90 minutes
**Difficulty:** Intermediate
**Format:** Individual
**Prerequisites:** [Activity 3](./Activity_3_Keyword_to_Semantic_Search.ipynb) complete

## The job

**RAG** stands for **R**etrieval-**A**ugmented **G**eneration: find the passages relevant to a question, hand them to a model as context, and let it answer from real source documents instead of whatever it absorbed during training.

You already built the retrieval half in Activity 3, over real, messy PDF text, three different ways. This notebook adds the generation half, and it is going to be less magical than you expect. There is no RAG library call at the center of this. There is a string you build yourself.

Then you will replace your by-hand pieces twice, and the point of doing it twice is to separate two things people constantly confuse:

- **Chroma** replaces your storage and search loop. That is a **database** problem.
- **LlamaIndex** replaces your loading, chunking, retrieval, and prompt assembly. That is a **framework** problem.

Knowing which layer does what is the difference between debugging a RAG system in ten minutes and staring at it for a day.

## What you will learn

- What "augmented" actually means, by building the augmented prompt by hand and reading it
- How to make an answer traceable, and how to test whether the model obeys its instructions
- Why `top_k` is a cost decision and a quality decision at the same time
- What a vector database adds beyond a `for` loop: persistence, filtering, and approximate search
- The LlamaIndex object model, mapped one to one onto the code you already wrote
- Why `SentenceSplitter` produces better chunks than the character counter you wrote, and the units trap inside it

---
## Setup

This is Activity 3's pipeline, verbatim, so this notebook stands on its own. You wrote every line of it already, so run it and move on.

In [ ]:
import os
import re
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader

load_dotenv()
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])


def extract_text(path):
    reader = PdfReader(path)
    return " ".join(page.extract_text() for page in reader.pages)


def chunk_text(text, chunk_size=800, overlap=100):
    chunks = []
    stride = chunk_size - overlap
    for start in range(0, len(text), stride):
        chunks.append(text[start:start + chunk_size])
    return chunks


PAPER_FILES = ["1508.07909v5.pdf", "1301.3781v3.pdf", "glove.pdf"]

CORPUS = []
for fname in PAPER_FILES:
    text = extract_text(f"pdfs/{fname}")
    for i, chunk in enumerate(chunk_text(text)):
        CORPUS.append({"id": f"{fname}#{i}", "source": fname, "text": chunk})

print(f"{len(CORPUS)} chunks from {len(PAPER_FILES)} papers")

In [ ]:
def embed_batch(texts, batch_size=100):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        response = client.embeddings.create(model="text-embedding-3-small", input=texts[i:i + batch_size])
        all_embeddings.extend(item.embedding for item in response.data)
    return all_embeddings


def cosine_similarity(a, b):
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


corpus_embeddings = embed_batch([doc["text"] for doc in CORPUS])


def semantic_search(query, top_k=3):
    query_vector = client.embeddings.create(model="text-embedding-3-small", input=query).data[0].embedding
    scored = [(doc, cosine_similarity(query_vector, v)) for doc, v in zip(CORPUS, corpus_embeddings)]
    return sorted(scored, key=lambda pair: pair[1], reverse=True)[:top_k]


print(f"{len(corpus_embeddings)} embeddings ready")

---
# 1. Start with the problem, not the solution

Before building RAG, watch what you get without it. Ask the model the same question from Activity 3, with no context attached.

In [ ]:
question = "What makes it possible to translate a word the model has never encountered?"

ungrounded = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": question}],
)
print(ungrounded.choices[0].message.content)

Read that answer critically, because the honest critique is narrower than people usually claim.

It is probably **not wrong**. This is a well-known topic, and the model likely mentioned subword units or byte pair encoding, which is genuinely the right answer. So "the model hallucinated" is not the problem here, and if you teach RAG as purely a hallucination fix you will misjudge when to use it.

The real problems are these:

1. **You cannot check it.** There is no source. To verify any sentence in that answer you would have to already know the answer.
2. **It knows nothing about your documents.** It answered from general training knowledge. Point it at your company's claims manuals or a set of papers published last month and that knowledge does not exist.
3. **You cannot tell which sentence came from where.** Even the correct parts have no provenance.

RAG fixes all three by changing the input, not the model.

---
# 2. Retrieval, unchanged from Activity 3

`semantic_search` already returns the right chunks for this question.

In [ ]:
retrieved = semantic_search(question, top_k=3)

for doc, score in retrieved:
    print(f"{score:.3f}  {doc['id']}")

Three chunks from `1508.07909v5.pdf`, the subword units paper, on a question that never uses that phrase. That is Activity 3's result, and it is the entire retrieval half of RAG.

You are using `semantic_search` here rather than the `hybrid_search` you finished Activity 3 with, purely so there is one less moving part while you learn the generation half. The "Your Turn" section asks you to swap it, and nothing else in the pipeline has to change, which is itself the point: retrieval is a pluggable component.

---
# 3. "Augmented" means you build a string

Here is the part that surprises people. There is no RAG API. The augmentation step is string concatenation, and you are about to do it in the least clever way possible so you can see exactly what the model receives.

In [ ]:
context = "\n\n".join(f"[{doc['id']}]\n{doc['text']}" for doc, score in retrieved)

print(context[:1200])
print("\n...\n")
print(f"total context length: {len(context):,} characters")

That is it. That is the augmentation: the retrieved chunks, labelled with their ids, glued together with newlines.

The labels matter more than they look. You are about to ask the model to cite its sources, and it can only cite an id if you put the id in front of the text. **A model cannot cite what you did not label.**

Now build the full message payload and look at that too, before sending it.

In [ ]:
SYSTEM_PROMPT = (
    "Answer using only the provided paper excerpts. "
    "Cite the chunk id(s) you used, in square brackets. "
    "If the excerpts do not answer the question, say so explicitly."
)

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": f"Paper excerpts:\n{context}\n\nQuestion: {question}"},
]

for m in messages:
    print(f"--- {m['role']} ({len(m['content']):,} chars) ---")
    print(m["content"][:300])
    print()

Two roles. The system message states the rules. The user message carries the evidence and the question together.

Note what is *not* happening: nothing was fine-tuned, nothing was trained, and the model has no memory of your corpus. Every RAG call re-sends the evidence. That has a direct cost, so measure it before you scale it.

In [ ]:
import tiktoken

encoder = tiktoken.encoding_for_model("gpt-4o-mini")

context_tokens = len(encoder.encode(context))
total_tokens = sum(len(encoder.encode(m["content"])) for m in messages)

print(f"context only : {context_tokens:,} tokens")
print(f"full request : {total_tokens:,} tokens")
print(f"question only: {len(encoder.encode(question)):,} tokens")

The question is a couple of dozen tokens. The evidence around it is two orders of magnitude larger. **In RAG, you are almost never paying for the question, you are paying for the context**, and that is the number that scales with `top_k`.

Section 6 turns that into actual money.

---
# 4. The grounded call

Same model, same question, different input.

In [ ]:
grounded = client.chat.completions.create(model="gpt-4o-mini", messages=messages)

print(grounded.choices[0].message.content)

Compare this against section 1. The content may be similar, since the ungrounded answer was already roughly right. What changed is that this answer carries chunk ids, and those ids point at text sitting in your `CORPUS` that you can go read.

That traceability, not eloquence, is the deliverable. When someone asks "where did that come from", you have a citation instead of a shrug. Verify one.

In [ ]:
cited_ids = re.findall(r"\[([^\]]+\.pdf#\d+)\]", grounded.choices[0].message.content)
print("model cited:", cited_ids)

for cid in dict.fromkeys(cited_ids):
    match = next((d for d in CORPUS if d["id"] == cid), None)
    print(f"\n--- {cid} {'FOUND' if match else 'DOES NOT EXIST'} ---")
    if match:
        print(match["text"][:300])

Run that and check two things: that every cited id actually exists in `CORPUS`, and that the text under it genuinely supports the claim.

Do not skip the first check. A model can invent a plausible-looking citation, and a citation that does not resolve is worse than no citation, because it looks like evidence. **In production, you validate cited ids against your corpus programmatically and reject answers that cite anything you did not send.** The cell above is the small version of that check.

---
# 5. Test the instruction, do not trust it

Your system prompt says "if the excerpts do not answer the question, say so." That is a hopeful sentence, not a guarantee. Find out whether it holds.

Your corpus is three papers about subword units, word2vec, and GloVe. None of them covers transformer attention, which was published later. So retrieval will return *something*, because retrieval always returns the nearest chunks whether or not they are relevant, and the model will have to decide.

In [ ]:
unanswerable = "How many attention heads does the transformer encoder use, and why that number?"

off_topic = semantic_search(unanswerable, top_k=3)
for doc, score in off_topic:
    print(f"{score:.3f}  {doc['id']}")

Look at those scores next to the ones in section 2. They are lower, but they are not zero and there is no obvious cutoff that says "nothing here". This is the same lesson as Activity 3 section 6.1: an absolute similarity score does not tell you whether a result is relevant.

So the safety net has to be the model, which is why the system prompt exists. Test it.

In [ ]:
off_context = "\n\n".join(f"[{doc['id']}]\n{doc['text']}" for doc, score in off_topic)

refusal_check = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Paper excerpts:\n{off_context}\n\nQuestion: {unanswerable}"},
    ],
)
print(refusal_check.choices[0].message.content)

Read what actually came back rather than assuming.

The behaviour you want is an explicit statement that the excerpts do not answer the question. The behaviour you do not want is a confident answer about attention heads assembled from general training knowledge, with the retrieved chunks quietly ignored, because that is a hallucination wearing a RAG costume.

Models are not perfectly reliable at this, and the same prompt can behave differently across runs and across models. That is exactly why Activity 6 builds a labelled evaluation set instead of checking one example by hand. **One passing test is an anecdote.** Write down which behaviour you observed, because you will compare against it there.

---
# 6. `top_k` is two decisions at once

`top_k` looks like a retrieval knob. It is also a cost knob and a quality knob, and the three pull against each other.

In [ ]:
# Prices per 1M tokens, gpt-4o-mini. Check the current numbers at
# https://platform.openai.com/docs/pricing, they change.
INPUT_PRICE_PER_1M = 0.150

# TODO: Fill in the loop body so this prints one row per top_k value.
#
# For each k:
#   1. Retrieve k chunks and build the context string, exactly the way
#      you did in section 3.
#   2. Count its tokens with encoder.encode(...).
#   3. Work out the cost of ONE query, then of 100,000 queries.
#
# Hint, the cost of n tokens: n / 1_000_000 * INPUT_PRICE_PER_1M
#       Prices are quoted per million tokens, so divide first.
# Hint: you only need the input price here. Output is a few hundred tokens
#       regardless of k, so it is not what top_k changes.
#
# Before you run it, predict: is the relationship between k and cost
# linear? Check whether the numbers agree with your prediction.

print(f"{'top_k':>6} {'chars':>8} {'tokens':>8} {'cost/query':>12} {'cost/100k queries':>19}")
for k in [1, 3, 5, 10, 20]:
    pass

<details>
<summary>Still stuck? Hint 1: building the context for a given k</summary>

This is the same line you already ran in section 3, with `top_k=k` instead of a fixed 3:

```python
ctx = "\n\n".join(f"[{d['id']}]\n{d['text']}" for d, s in semantic_search(question, top_k=k))
```

Remember `semantic_search` returns `(doc, score)` pairs, which is why the comprehension unpacks two values and only uses the first.

</details>

<details>
<summary>Still stuck? Hint 2: the token count and the arithmetic</summary>

```python
n_tokens = len(encoder.encode(ctx))
per_query = n_tokens / 1_000_000 * INPUT_PRICE_PER_1M
```

Then print `k`, `len(ctx)`, `n_tokens`, `per_query`, and `per_query * 100_000`.

If your per-query number shows as `0.00000`, that is not a bug, it is a real result: widen the format to five decimal places (`:.5f`) so you can see it, and note that this is exactly why the 100,000-query column exists.

</details>

Per query, every one of those numbers is negligible. That is the trap: RAG feels free while you are testing it in a notebook, so nobody thinks about `top_k` as a spending decision.

Look at the multiple instead of the absolute. Going from `top_k=3` to `top_k=20` costs about seven times as much, on every single query, forever. Apply that multiple to a corpus of real documents, a longer chunk size, a more expensive model, and a support desk's query volume, and it stops being a rounding error. The absolute numbers here are small because the demo is small; the ratio is what transfers.

Cost is only half of it. More context is not monotonically better quality either:

- **Too few chunks** and the answer is simply not in the context. No prompt can recover it. This is a **retrieval** failure, and no amount of prompt engineering fixes it.
- **Too many chunks** and the relevant passage is buried among near misses. Models attend unevenly to long contexts, with a well-documented tendency to use the beginning and end more reliably than the middle, so a correct chunk sitting at position 14 of 20 may be effectively invisible.

The practical consequence is that "just retrieve more" is not a safe default, and you cannot pick `top_k` by reasoning about it. You pick it by measuring, which is Activity 6.

---
# 7. Package it

You now have every piece. Wrap them into one function.

In [ ]:
# TODO: Write rag_answer(question, top_k=3).
#
# Do the four steps you just did by hand, in order:
#   1. Retrieve top_k chunks with semantic_search.
#   2. Build the labelled context string, same format as section 3:
#      "[id]\ntext", chunks joined by a blank line.
#   3. Call gpt-4o-mini with SYSTEM_PROMPT and the excerpts-plus-question
#      user message.
#   4. Return a dict:
#        {"answer": <the text>, "sources": [<chunk ids used as context>]}
#
# Return the sources, not just the answer. An answer you cannot trace is
# the thing this whole notebook exists to avoid, and a caller that only
# gets a string has no way to show a user where it came from.
#
# Hint: retrieved = semantic_search(question, top_k=top_k) gives you
#       (doc, score) pairs, so doc["id"] and doc["text"] are what you need.
# Hint: response.choices[0].message.content is the answer text.

def rag_answer(question, top_k=3):
    pass


result = rag_answer("How does GloVe combine global matrix factorization with local context windows?")
print(result["answer"])
print()
print("sources:", result["sources"])

<details>
<summary>Still stuck? Hint 1: the shape of the function</summary>

Every piece already exists somewhere above. Steps 1 and 2 are sections 2 and 3, unchanged:

```python
def rag_answer(question, top_k=3):
    retrieved = semantic_search(question, top_k=top_k)
    context = "\n\n".join(f"[{doc['id']}]\n{doc['text']}" for doc, score in retrieved)
    # ... the API call goes here ...
    return {"answer": ..., "sources": [doc["id"] for doc, score in retrieved]}
```

Note that `retrieved` is used twice: once to build the context, once to report the sources. That is why it goes in its own variable instead of being inlined.

</details>

<details>
<summary>Still stuck? Hint 2: the API call</summary>

Copy the `messages` structure from section 3 and swap the hardcoded values for the function's arguments:

```python
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Paper excerpts:\n{context}\n\nQuestion: {question}"},
    ],
)
```

The answer text is `response.choices[0].message.content`.

</details>

<details>
<summary>Still stuck? Hint 3: it runs but sources is empty or wrong</summary>

`sources` must be built from the same `retrieved` list you used for the context, not from anything the model returned. You are reporting **what you sent**, which is a fact you control, not what the model claims it used, which it can get wrong.

Section 4 checked the model's claimed citations separately, and those two things are different on purpose.

</details>

That is a complete RAG pipeline in about a dozen lines, and you understand every one of them. Everything after this point replaces parts of it with something faster or shorter, and nothing after this point does anything conceptually new.

---
# 8. Where the by-hand version breaks: the storage layer

Look honestly at what `semantic_search` does on every single call.

In [ ]:
import time

start = time.time()
query_vector = client.embeddings.create(model="text-embedding-3-small", input=question).data[0].embedding
api_time = time.time() - start

start = time.time()
scored = [(doc, cosine_similarity(query_vector, v)) for doc, v in zip(CORPUS, corpus_embeddings)]
scored.sort(key=lambda pair: pair[1], reverse=True)
loop_time = time.time() - start

print(f"embedding the query (network) : {api_time:.3f}s")
print(f"scoring {len(CORPUS)} chunks (your loop): {loop_time:.3f}s")

Time the two halves separately and the picture is the opposite of what you might guess. The scoring loop is nearly instant. Almost all of that wall-clock time is one network round trip to the embeddings endpoint.

So do not conclude "my loop is slow". Conclude that **the loop is not the bottleneck yet**, and understand what changes when it becomes one. Scoring is `O(n)`: at 189 chunks it is invisible, at 5 million chunks it dominates completely and the API call becomes the cheap part.

Three specific things are wrong with the by-hand version:

**It is exact and linear.** The fix at scale is not a faster loop, it is a different algorithm: **approximate nearest neighbour** search, which trades a small amount of recall for a very large speedup by not comparing against most of the corpus at all.

**Nothing is persisted.** `corpus_embeddings` lives in a Python variable. Restart the kernel and you pay the embedding bill again. You already have `embed_batch`, so this is your money.

**You cannot filter.** "Search only the GloVe paper" is not expressible. You would have to rebuild a filtered `CORPUS` and re-run everything.

A **vector database** is exactly these three things packaged: approximate search, persistence, and metadata filtering.

In [ ]:
import chromadb

chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name="paper_chunks")

collection.add(
    ids=[doc["id"] for doc in CORPUS],
    documents=[doc["text"] for doc in CORPUS],
    embeddings=corpus_embeddings,
    metadatas=[{"source": doc["source"]} for doc in CORPUS],
)
print(collection.count(), "chunks indexed")

Note what was passed in: ids, documents, **embeddings you already computed**, and metadata. Chroma will happily embed text for you, but passing your own vectors keeps this identical to what `semantic_search` searches, so any difference in results comes from the search, not from a different embedding model.

The `metadatas` argument is the new capability. Each chunk is now tagged with its source paper.

In [ ]:
query_vector = client.embeddings.create(model="text-embedding-3-small", input=question).data[0].embedding

results = collection.query(query_embeddings=[query_vector], n_results=3)

for chunk_id, meta, distance in zip(results["ids"][0], results["metadatas"][0], results["distances"][0]):
    print(f"{distance:.3f}  {chunk_id}  ({meta['source']})")

Same chunks as section 2, in the same order, because it is the same vectors and the same math. Chroma did the loop and the sort.

One thing did change: these are **distances**, not similarities. Lower is better here, where higher was better before. Every vector store makes its own choice about this, and mixing the two up silently reverses your ranking, which is a bug that produces confident nonsense rather than an error.

Now the capability you did not have.

In [ ]:
# TODO: Query the collection for the same question, but restricted to
# glove.pdf only.
#
# Chroma takes a `where` argument that filters on the metadata you
# supplied in collection.add:
#
#     where={"source": "glove.pdf"}
#
# Run it, then print each result's chunk id and source, and confirm every
# result comes from glove.pdf.
#
# Then ask yourself the question that matters: what would you have had to
# do to get this same behaviour out of semantic_search?

filtered = None

<details>
<summary>Still stuck? Hint: the filtered query</summary>

It is the same `collection.query` call from the cell above with one extra argument:

```python
filtered = collection.query(
    query_embeddings=[query_vector],
    n_results=3,
    where={"source": "glove.pdf"},
)
```

Then print the results the same way:

```python
for chunk_id, meta, distance in zip(filtered["ids"][0], filtered["metadatas"][0], filtered["distances"][0]):
    print(f"{distance:.3f}  {chunk_id}  ({meta['source']})")
```

The `[0]` on each list is because Chroma supports batching several queries at once, so it always returns a list of result sets. You sent one query, so you want the first.

If you get an empty result, check that the key in `where` matches the metadata key you passed to `collection.add`, which was `"source"`, and that the value matches a filename exactly.

</details>

Every result from one paper, with the retrieval logic untouched. In a real system that same mechanism is how you scope a search to one customer, one policy year, or one document type, and it is a hard requirement in most regulated settings rather than a nice extra.

**What Chroma did not replace:** loading, chunking, prompt assembly, or the generation call. Those are still your code. That is the boundary of the database layer, and it is exactly where the framework layer starts.

---
# 9. LlamaIndex: the framework layer

**LlamaIndex** is a framework built specifically around this load, chunk, embed, store, retrieve, generate pattern. Its value for you right now is not that it is short. It is that its object model has a name for every piece you just built by hand:

| What you wrote | LlamaIndex calls it |
| :--- | :--- |
| `extract_text` output, one per paper | `Document` |
| A dict in `CORPUS` | `Node` |
| `chunk_text` | `NodeParser` (here, `SentenceSplitter`) |
| `corpus_embeddings` plus Chroma | `VectorStoreIndex` |
| `semantic_search` | `Retriever` |
| Building the context string in section 3 | `ResponseSynthesizer` |
| `rag_answer` | `QueryEngine` |
| Your `hybrid_search` from Activity 3 | `QueryFusionRetriever` |

Read that table twice. You are not learning new concepts here, you are learning the standard names for concepts you already implemented. That is why the by-hand version came first.

In [ ]:
from llama_index.core import Document, VectorStoreIndex, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI as LlamaOpenAI

Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")
Settings.llm = LlamaOpenAI(model="gpt-4o-mini")

print("embed model:", Settings.embed_model.model_name)
print("llm        :", Settings.llm.model)

`Settings` is a global default. Set the models once and every component below uses them, instead of threading a model name through every call. Convenient, and worth knowing about, because a component quietly using a default you forgot to set is a classic source of "why is this answer different".

Now build `Document` objects. You could point LlamaIndex at the folder and let it read the PDFs, but you are going to reuse your own `extract_text` instead, for two reasons: the mapping to what you already built stays visible, and this repository pins a `pandas` version that conflicts with LlamaIndex's file-reader package, which is a real dependency constraint of the kind you met with MLflow in Activity 0.

In [ ]:
documents = [
    Document(text=extract_text(f"pdfs/{fname}"), metadata={"source": fname})
    for fname in PAPER_FILES
]

print(f"{len(documents)} Documents")
print("metadata:", documents[0].metadata)
print("characters:", f"{len(documents[0].text):,}")

Three `Document` objects, one per paper, each carrying metadata. Same content as `extract_text` produced in the setup cell, in a container the framework understands.

## 9.1 Chunking, and a units trap worth knowing about

`SentenceSplitter` is the smarter splitter Activity 3 promised you. It tries to break on sentence boundaries and only falls back to a hard cut when it has to.

It also has a trap in it. Set `chunk_size=800`, the same number you used in Activity 3, and look at what you get.

In [ ]:
trap = SentenceSplitter(chunk_size=800, chunk_overlap=100).get_nodes_from_documents(documents)
lengths = [len(n.text) for n in trap]

print(f"{len(trap)} nodes")
print(f"characters per node: min {min(lengths)}, mean {sum(lengths) // len(lengths)}, max {max(lengths)}")

You asked for 800 and got nodes averaging well over 2,000 characters. Nothing is broken. **`SentenceSplitter` measures in tokens, `chunk_text` measured in characters**, and for English prose a token is roughly 4 characters.

This is the kind of detail that silently changes a system's behaviour. Someone migrating Activity 3's pipeline to LlamaIndex, keeping "the same" chunk size of 800, would quietly triple their chunk sizes, triple their per-query context cost, and wonder why retrieval got less precise.

**Always check the units on a size parameter.** Use about 200 tokens to match Activity 3's 800 characters.

In [ ]:
splitter = SentenceSplitter(chunk_size=200, chunk_overlap=25)
nodes = splitter.get_nodes_from_documents(documents)

lengths = [len(n.text) for n in nodes]
print(f"{len(nodes)} nodes")
print(f"characters per node: min {min(lengths)}, mean {sum(lengths) // len(lengths)}, max {max(lengths)}")
print()
print("metadata carried down from the Document:", nodes[10].metadata)

Note that node lengths now **vary**, where every chunk in `CORPUS` was exactly 800 characters. That is the splitter respecting sentence boundaries instead of counting to a number.

It also means BM25's length normalization, the `b` parameter that Activity 3 showed was doing nothing on your fixed-size chunks, now has real work to do.

Compare the boundaries directly against your by-hand chunker.

In [ ]:
print("=== Activity 3, chunk_text (fixed characters) ===")
print(repr(CORPUS[10]["text"][:110]))
print("   ...ENDS:", repr(CORPUS[10]["text"][-70:]))
print()
print("=== LlamaIndex, SentenceSplitter ===")
print(repr(nodes[10].text[:110]))
print("   ...ENDS:", repr(nodes[10].text[-70:]))

The fixed-size chunk starts and ends wherever the character count ran out. The `SentenceSplitter` node is far more likely to begin and end on a real sentence boundary.

Be fair about the limits, though: this splitter is boundary-aware, not structure-aware. It does not know what a table, a code listing, or a bibliography is, and these papers contain all three. Some nodes will still start in the middle of a formula.

## 9.2 Index, retriever, query engine

Three lines to replace section 8's Chroma setup and section 2's search.

In [ ]:
index = VectorStoreIndex(nodes)
print("index built")

In [ ]:
retriever = index.as_retriever(similarity_top_k=3)

for node in retriever.retrieve(question):
    print(f"{node.score:.3f}  {node.metadata['source']}  ::  {node.text[:80]}...")

That is `semantic_search`. It embedded the query, scored it against every node, sorted, and returned the top 3 with scores attached, and the right paper came back.

Now the generation half.

In [ ]:
query_engine = index.as_query_engine(similarity_top_k=3)
response = query_engine.query(question)

print(response)
print()
print("--- sources ---")
for node in response.source_nodes:
    print(f"{node.score:.3f}  {node.metadata['source']}")

That is `rag_answer`: retrieve, build a prompt, call the model, return an answer with its sources attached. The `response_nodes` are the citation trail, handed to you as objects instead of ids you have to parse out of the text with a regex the way you did in section 4.

Everything from sections 2 through 7 is in those three lines. That compression is what a framework is for, and it is also why doing it by hand first was worth the time: when this returns something wrong, you know it is a retrieval problem or a synthesis problem, and you know where to go look.

## 9.3 The payoff: your hybrid search, as a named component

Activity 3 ended with you hand-implementing Reciprocal Rank Fusion over BM25 and embeddings, because each one failed on query types the other handled.

That algorithm has a name and a class. Build both retrievers, then fuse them.

In [ ]:
from llama_index.retrievers.bm25 import BM25Retriever

vector_retriever = index.as_retriever(similarity_top_k=10)
bm25_retriever = BM25Retriever.from_defaults(nodes=nodes, similarity_top_k=10)

print("--- vector retriever, 'Morfessor' ---")
for n in vector_retriever.retrieve("Morfessor")[:3]:
    print(f"  {n.score:.3f}  {n.text[:70]!r}")

print("\n--- BM25 retriever, 'Morfessor' ---")
for n in bm25_retriever.retrieve("Morfessor")[:3]:
    print(f"  {n.score:.3f}  {n.text[:70]!r}")

The same split you found in Activity 3, reproduced with library components on differently-chunked text. Confirm which node actually contains the word, so you are judging against ground truth rather than impressions.

In [ ]:
for i, n in enumerate(nodes):
    if "Morfessor" in n.text:
        print(f"node {i} contains it:", n.text[:100].replace("\n", " "), "...")

In [ ]:
# TODO: Build a QueryFusionRetriever that fuses vector_retriever and
# bm25_retriever with Reciprocal Rank Fusion, and retrieve "Morfessor".
#
# from llama_index.core.retrievers import QueryFusionRetriever
#
# Arguments you need:
#   [vector_retriever, bm25_retriever]   the retrievers to fuse, as a list
#   similarity_top_k=3                   how many results to return
#   num_queries=1                        IMPORTANT, see below
#   mode="reciprocal_rerank"             this is the RRF you wrote by hand
#   use_async=False                      keeps it simple inside a notebook
#
# About num_queries: LlamaIndex defaults to 4, which means it asks an LLM
# to rewrite your query into several variations and fuses across all of
# them too. That is a genuinely useful technique, but it is a SECOND thing
# happening at the same time, it costs an extra LLM call per search, and it
# would make it impossible to tell whether an improvement came from fusion
# or from query rewriting. Set it to 1 so you are testing one idea.
#
# Expected: the node you just confirmed contains "Morfessor" should rank
# FIRST, even though neither retriever alone put it first.

fusion_retriever = None

for n in fusion_retriever.retrieve("Morfessor"):
    print(f"{n.score:.4f}  {n.text[:80]!r}")

<details>
<summary>Still stuck? Hint 1: the import and the constructor</summary>

```python
from llama_index.core.retrievers import QueryFusionRetriever

fusion_retriever = QueryFusionRetriever(
    [vector_retriever, bm25_retriever],
    similarity_top_k=3,
    num_queries=1,
    mode="reciprocal_rerank",
    use_async=False,
)
```

The retrievers go in as a plain list, as the first positional argument.

</details>

<details>
<summary>Still stuck? Hint 2: it runs but the results look wrong</summary>

Check `num_queries` first. If you left it at the default of 4, LlamaIndex is generating extra rewritten versions of your query with an LLM and fusing across those too, so you are no longer measuring what fusion alone does, and every retrieval costs an extra model call.

Also check that `vector_retriever` and `bm25_retriever` were built with `similarity_top_k=10`, not 3. Fusion needs a deeper candidate list from each retriever than it returns, otherwise a node ranked 4th by one method and 5th by the other can never surface, which is exactly the kind of broad-agreement result you want fusion to find.

</details>

<details>
<summary>Still stuck? Hint 3: what counts as success here</summary>

Do not judge this by whether the scores look nice. The fused scores are small numbers like `0.03`, because RRF sums `1 / (60 + rank)` terms, and they are not comparable to either input scale.

The test is positional: the node you confirmed contains "Morfessor" in the cell above should now be **first**, when the vector retriever had it third and BM25 had it second.

</details>

Check the top result against the node you confirmed contains "Morfessor".

Neither retriever ranked that node first on its own. The vector retriever had it buried below chunks that merely looked like a bibliography, and BM25 ranked another chunk above it. Fusion put it first, because it was the one node **both** methods liked, and that is precisely the behaviour RRF is designed to produce. A document with broad agreement beats a document one ranker loved and the other ignored.

You wrote this in Activity 3 in about eight lines. Here it is a class with a `mode` argument. The eight lines are why you can now read that argument and know exactly what it does.

In [ ]:
from llama_index.core.query_engine import RetrieverQueryEngine

hybrid_engine = RetrieverQueryEngine.from_args(fusion_retriever)
hybrid_response = hybrid_engine.query(question)

print(hybrid_response)
print()
print("sources:", [n.metadata["source"] for n in hybrid_response.source_nodes])

A full hybrid RAG pipeline: BM25 plus embeddings, fused by RRF, feeding a grounded generation call. In Activity 3 and this notebook you built every one of those pieces yourself first.

---
# 10. The same thing in LangChain

**LangChain** is the other framework you will meet, and you are much more likely to inherit LangChain code at work than LlamaIndex code. It is worth seeing the same five steps in its vocabulary so the code does not look foreign later.

You may see a `DeprecationWarning` from `langchain_community` below. That package is in reduced-maintenance mode while LangChain moves integrations into smaller, dedicated packages. `PyPDFLoader` still works and is still the documented way to load a PDF.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

pages = PyPDFLoader("pdfs/1508.07909v5.pdf").load()
print(f"{len(pages)} pages loaded")
print("page 0 metadata:", pages[0].metadata)

One page, one `Document`, tagged with its page number for free. That is a real difference from your by-hand pipeline and from the LlamaIndex path above, both of which flattened each paper into one string and lost page numbers. If your users ask "which page", that metadata is the whole answer.

In [ ]:
lc_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
lc_chunks = lc_splitter.split_documents(pages)

print(f"{len(lc_chunks)} chunks")
print("first chunk:", repr(lc_chunks[5].page_content[:150]))

Note that `RecursiveCharacterTextSplitter` measures in **characters**, so 800 here means what 800 meant in Activity 3, and it means something different from `SentenceSplitter(chunk_size=800)`. Two frameworks, same parameter name, different units. This is not a trick question, it is Tuesday.

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

lc_store = Chroma.from_documents(
    lc_chunks,
    embedding=OpenAIEmbeddings(model="text-embedding-3-small"),
    collection_name="bpe_paper_langchain",
)

for doc in lc_store.similarity_search(question, k=3):
    print(f"page {doc.metadata['page']}: {doc.page_content[:90]}...")

Load, split, embed, store, search. Same five steps, different nouns.

---
# 11. Choosing

| | By hand | Chroma | LlamaIndex | LangChain |
| :--- | :--- | :--- | :--- | :--- |
| Lines to a working pipeline | ~40 | ~30 | ~5 | ~8 |
| You control chunking | Fully | Fully | Configurable | Configurable |
| Persistence | No | Yes | Via a vector store | Via a vector store |
| Metadata filtering | No | Yes | Yes | Yes |
| Hybrid search | You write RRF | Not included | `QueryFusionRetriever` | `EnsembleRetriever` |
| Page-level metadata | You add it | You add it | Via its file readers | Free with `PyPDFLoader` |
| Debuggable when wrong | Completely | Mostly | Layer by layer | Layer by layer |

**Which framework?** For document question answering, which is this lab and a large share of real RAG work, LlamaIndex fits better. Its abstractions are named after retrieval concepts, so the code reads like the diagram, and things like fusion retrieval and node post-processing are first-class rather than assembled.

LangChain is the broader toolkit and has the larger installed base, especially once a project grows past retrieval into agents and multi-step orchestration, where LangGraph is the stronger story. You will meet it more often in existing codebases.

The honest answer is that this choice matters far less than people arguing about it suggest. Both wrap the same five steps around the same embedding models and the same vector stores. Retrieval quality is decided by your chunking, your embedding model, and your evaluation, none of which is a framework feature.

**Which layer?** That distinction is worth more than the framework choice:

- Answers are wrong or irrelevant, fix **retrieval**: chunking, `top_k`, hybrid search.
- Answers are slow, or you re-embed on every restart, fix **storage**: a real vector database.
- The code is unmaintainable, fix **the framework layer**, and only then.

Reaching for a framework when the problem is retrieval quality is the single most common mistake in this space. A framework will make your bad retrieval shorter, not better.

---
# Your Turn

Work in your own copy under `student-work/week6/day4/`.

1. **Swap the retriever.** Copy your `hybrid_search` from Activity 3 into this notebook and change `rag_answer` to use it instead of `semantic_search`. Nothing else should need to change, which is the point. Then ask it "Morfessor" and compare against the `semantic_search` version.

2. **Break the grounding on purpose.** Change `SYSTEM_PROMPT` to remove the "if the excerpts do not answer the question, say so" instruction, then re-run the section 5 unanswerable question. Write down what changed. This is the cheapest possible demonstration of why that sentence is in there.

3. **Make the citation check strict.** Extend section 4's verification into a function that returns `False` if the model cites any id not present in `CORPUS`. Then try to get it to fail: ask something the corpus half-answers, and see whether it invents an id.

4. **Find the `top_k` cliff.** Ask a question whose answer sits in exactly one chunk. Run `rag_answer` at `top_k=1`, `3`, and `10`. Find the value where the answer stops being correct, and say whether the failure was retrieval or synthesis.

**Stretch goal:** persist the Chroma collection to disk with `chromadb.PersistentClient(path=...)`, restart your kernel, and reload it without re-embedding anything. Time both paths and calculate what you saved. That saving is the entire argument for a vector database in one number.

## What you did

- Watched an ungrounded answer that was not wrong, and worked out why that is still not good enough.
- Built the augmented prompt by hand, and saw that "augmentation" is labelled string concatenation.
- Measured the context in tokens and money, and found that `top_k` is a cost decision as much as a quality one.
- Made answers traceable, then verified the citations actually resolve rather than trusting them.
- Tested whether the model obeys its "say so" instruction instead of assuming it, on a question the corpus cannot answer.
- Replaced your search loop with Chroma and gained persistence, approximate search, and metadata filtering.
- Rebuilt the whole pipeline in LlamaIndex, mapping every object back to code you had already written, and found the units trap in `SentenceSplitter`.
- Reproduced Activity 3's hand-written RRF as `QueryFusionRetriever`, and watched fusion rank the correct node first when neither retriever alone did.
- Saw the same five steps once more in LangChain, and built a basis for choosing between them.

**Next:** [Activity 5](./Activity_5_MLflow_Tour.ipynb) is a short detour into MLflow, before Activity 6 uses it to evaluate a RAG pipeline properly, with a labelled eval set instead of the hand-picked questions you used here.